---
# Section 1: Pre-processing
---

Flow: đọc 4 table

I. Làm sạch dữ liệu:
1. categories
2. products
3. reviews
4. stores

mỗi mục:
- thống kê cơ bản (dimensions, info)
- kiểm tra trùng
- kiểm tra missing
- kiểm tra unique từng thuộc tính (mục đích: tìm rác)

II. Chuẩn hóa dữ liệu:
1. kiểm tra mapping (các trường dữ liệu giữa các bảng có join nhau được không) -> tìm data không mapping được để sau này tạo database lưu ý
2. kiểm tra xem có sp nào thuộc nhiều hơn 1 category k
3. kiểm tra category (xem đã phân loại product đúng theo category chưa) -> dùng các phương pháp để xử lý nếu có dữ liệu sai category (vd: tf-idf)

## Đọc dữ liệu và khám phá sơ bộ

Các bước thực hiện: 
- Đọc 4 bảng dữ liệu: categories, products, reviews, stores.
- Thực hiện thống kê cơ bản, kiểm tra trùng, thiếu và giá trị unique cho từng bảng.
- Kiểm tra khả năng ánh xạ giữa các bảng.
- Kiểm tra lại phân loại category của sản phẩm ở mức đơn giản.

### Đọc dữ liệu

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


In [2]:
# Đường dẫn thư mục data (tương đối so với notebook)
data_dir = Path("../data")

categories_path = data_dir / "categories.csv"
products_path = data_dir / "products.csv"
reviews_path = data_dir / "reviews.csv"
stores_path = data_dir / "stores.csv"

print("Đọc dữ liệu từ:")
print(f"- categories: {categories_path}")
print(f"- products:   {products_path}")
print(f"- reviews:    {reviews_path}")
print(f"- stores:     {stores_path}")

df_category = pd.read_csv(categories_path)
df_product = pd.read_csv(products_path)
df_review = pd.read_csv(reviews_path)
df_store = pd.read_csv(stores_path)

Đọc dữ liệu từ:
- categories: ..\data\categories.csv
- products:   ..\data\products.csv
- reviews:    ..\data\reviews.csv
- stores:     ..\data\stores.csv


In [3]:
df_category.head(5)

,category_id,category_name,parent_category,source_category
0,1882,Dien gia dung,NaN,diengiadung
1,1884,Đồ dùng nhà bếp,Dien gia dung,diengiadung
2,1892,Nồi điện các loại,Đồ dùng nhà bếp,diengiadung
3,1893,Nồi cơm điện,Nồi điện các loại,diengiadung
4,1918,Nồi cơm điện tử,Nồi cơm điện,diengiadung


In [4]:
df_product.head(5)

,product_id,store_id,category_id,product_name,product_url,brand,description,price,original_price,discount_percent,sold_count,rating_avg,review_count,source_category
0,273634419,1.0,1918,Nồi cơm điện tử Elmich 0.8L RCE-3915 - Hàng Ch...,https://tiki.vn/noi-com-dien-tu-elmich-0-8l-rc...,Elmich,Thông tin sản phẩm ...,1000000,2200000,55,130,5.0,18,diengiadung
1,134550148,1.0,1918,Nồi cơm điện tử Elmich RCE-1789 (1.2 Lít) - Hà...,https://tiki.vn/noi-com-dien-tu-elmich-rce-178...,Elmich,Chất liệu tuyệt đối cho sức khỏeNồi cơm điện t...,1136000,2500000,55,301,4.8,63,diengiadung
2,274106659,1.0,1918,Nồi Cơm Điện Tử Mini Philips HD3170/66 - 600W ...,https://tiki.vn/noi-com-dien-tu-mini-philips-h...,Philips,Nồi cơm điện Dòng 3000 của Philips có dung tíc...,965000,1290000,25,61,4.8,8,diengiadung
3,274965977,1.0,1918,"Nồi cơm điện tử Kangaroo 1.8L model KG18DR12, ...",https://tiki.vn/noi-com-dien-tu-1-8l-model-kg1...,Kangaroo,"Dung tích 1,8LDung tích nồi cơm điện KG18DR12 ...",1210000,2520000,52,16,4.8,5,diengiadung
4,260224713,1.0,1918,Nồi cơm điện cao tần Daewoo DWRC-G411IH 1.8L l...,https://tiki.vn/noi-com-dien-cao-tan-1-8l-daew...,Daewoo,...,1626000,2990000,46,9,3.0,2,diengiadung


In [5]:
df_review.head(5)

,review_id,product_id,user_name,rating,review_text,like_count,review_date,source_category
0,20007421,273634419,Hồ Hữu Nghĩa,5,"Sản phẩm xài rất ok. Nấu nhanh, gọn nhẹ, nhiểu...",0,2024-10-16T03:30:42.000Z,diengiadung
1,20205429,273634419,Le On,5,Cực kì hài lòng,0,2026-01-13T06:29:08.000Z,diengiadung
2,20202180,273634419,Luyen pv,5,Cực kì hài lòng,0,2025-12-30T16:02:30.000Z,diengiadung
3,20191102,273634419,Nguyễn Văn Khang,5,Cực kì hài lòng,0,2025-11-16T08:24:31.000Z,diengiadung
4,20149003,273634419,Hoàng Văn Chương,5,Dùng tốt,0,2025-07-20T07:46:41.000Z,diengiadung


In [6]:
df_store.head(5)

,store_id,store_name,store_rating,follower_count,source_category
0,1,Tiki Trading,4.6754,512930,diengiadung
1,308020,GiaDungNhaThongMinh,4.8750,39,diengiadung
2,29960,CUCKOO Store,4.6289,4894,diengiadung
3,160749,Mishio Kachi Official,4.5030,5396,diengiadung
4,360957,Seka Official Store,4.0000,0,diengiadung


## I. Làm sạch dữ liệu

Cho từng bảng `df_category`, `df_product`, `df_review`, `df_store`:
- Thống kê kích thước, kiểu dữ liệu
- Kiểm tra trùng dòng
- Kiểm tra missing
- Lọc dữ liệu.

### `df_category`

#### 1. Thống kê cơ bản

In [7]:
df_category.shape

(497, 4)

In [8]:
df_category.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 497 entries, 0 to 496
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   category_id      497 non-null    int64 
 1   category_name    497 non-null    object
 2   parent_category  492 non-null    object
 3   source_category  497 non-null    object
dtypes: int64(1), object(3)
memory usage: 15.7+ KB


Nhận xét ngắn:
- kiểu dữ liệu ntn
- cần ktra gì

#### 2. Check duplicate

In [9]:
df_category[df_category.duplicated(subset=["category_name"], keep=False)]

,category_id,category_name,parent_category,source_category
121,5976,Máy sấy quần áo,Thiết bị gia đình,diengiadung
166,3863,Máy sấy quần áo,Dien tu - Dien lanh,dientu_dienlanh


Nhóm sẽ kiểm tra dữ liệu ở 2 mục này sau để quyết định giữ category ở bên nào

In [10]:
# in 5 sản phẩm của category "Máy sấy quần áo" ở category_id = 5976
df_product[df_product["category_id"] == 5976].head(5)

,product_id,store_id,category_id,product_name,product_url,brand,description,price,original_price,discount_percent,sold_count,rating_avg,review_count,source_category
11098,278872428,1.0,5976,Tủ sấy quần áo Elmich CDE-8641 - Hàng Chính Hãng,https://tiki.vn/tu-say-quan-ao-elmich-cde-8641...,Elmich,Kích thước rộng rãi: Tủ sấy có kích thước tổng...,1358000,2988000,55,27,5.0,2,diengiadung
11099,278893185,1.0,5976,Tủ sấy quần áo Elmich CDE-1296 - Hàng Chính Hãng,https://tiki.vn/tu-say-quan-ao-elmich-cde-1296...,Elmich,Câu Chuyện Sản Phẩm Tủ sấy quần áo Elmich CDE1...,1540000,3388000,55,3,0.0,0,diengiadung
11100,279165505,292665.0,5976,"Tủ Sấy Quần Áo Elmich CDE1296, Hàng Chính Hãng...",https://tiki.vn/tu-say-quan-ao-elmich-cde1296-...,Elmich,"Tủ Sấy Quần Áo Elmich CDE1296, Hàng Chính Hãng...",2550000,2550000,0,0,0.0,0,diengiadung
11101,278989424,1.0,5976,"Máy sấy khuẩn quần áo LocknLocK,khử khuẩn tia ...",https://tiki.vn/may-say-khuan-quan-ao-locknloc...,LocknLock,Máy sấy khuẩn quần áo LocknLock Cloth steriliz...,1389500,1985000,30,0,0.0,0,diengiadung
11102,278877812,297289.0,5976,Tủ sấy quần áo KORENO KN - 569 LOẠI TO công su...,https://tiki.vn/tu-say-quan-ao-koreno-kn-569-l...,KORENO,CÁC ĐẶC ĐIỂM NỔI BẬT:- Khối lượng sấy quần áo ...,1650000,1650000,0,0,0.0,0,diengiadung


In [33]:
# in 5 sản phẩm của category "Máy sấy quần áo" ở category_id = 3863
df_product[df_product["category_id"] == 3863].head(5)

,product_id,store_id,category_id,product_name,product_url,brand,description,price,original_price,discount_percent,sold_count,rating_avg,review_count,source_category


Ta nhận thấy "Máy sấy quần áo" ở `category_id` = 3863 thực chất là một category rỗng

-> Tiến hành loại bỏ

In [34]:
# xóa category_id = 3863
df_category = df_category[df_category["category_id"] != 3863].copy()

In [35]:
#kiểm tra lại
df_category[df_category.duplicated(subset=["category_name"], keep=False)]

,category_id,category_name,parent_category,source_category


#### 3. Check missing

In [12]:
# kiểm tra missing toàn bộ thuộc tính
print("Missing values in df_category:")
print(df_category.isnull().sum())

Missing values in df_category:
category_id        0
category_name      0
parent_category    5
source_category    0
dtype: int64


Đây là 5 root category (5 nhánh thư mục gốc, không cần loại xử lý)

#### 4. Lọc dữ liệu

### `df_product`

#### 1. Thống kê cơ bản

In [13]:
df_product.shape

(55883, 14)

In [14]:
df_product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55883 entries, 0 to 55882
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        55883 non-null  int64  
 1   store_id          55023 non-null  float64
 2   category_id       55883 non-null  int64  
 3   product_name      55883 non-null  object 
 4   product_url       55883 non-null  object 
 5   brand             55881 non-null  object 
 6   description       55830 non-null  object 
 7   price             55883 non-null  int64  
 8   original_price    55883 non-null  int64  
 9   discount_percent  55883 non-null  int64  
 10  sold_count        55883 non-null  int64  
 11  rating_avg        55883 non-null  float64
 12  review_count      55883 non-null  int64  
 13  source_category   55883 non-null  object 
dtypes: float64(2), int64(7), object(5)
memory usage: 6.0+ MB


Nhận xét ngắn:
- kiểu dữ liệu ntn
- cần ktra gì

#### 2. Check duplicate

In [15]:
# kiểm tra product_url và brand và price có bị trùng lặp không
df_product[df_product.duplicated(subset=["product_url"], keep=False)]

,product_id,store_id,category_id,product_name,product_url,brand,description,price,original_price,discount_percent,sold_count,rating_avg,review_count,source_category


Không có trùng lặp

#### 3. Check missing

In [16]:
# kiểm tra missing
print("Missing values in df_product:")
print(df_product.isnull().sum())

Missing values in df_product:
product_id            0
store_id            860
category_id           0
product_name          0
product_url           0
brand                 2
description          53
price                 0
original_price        0
discount_percent      0
sold_count            0
rating_avg            0
review_count          0
source_category       0
dtype: int64


Các dữ liệu missing:
- `store_id`: không thu thập được dữ liệu store của các sản phẩm.
- `brand`: sản phẩm NoBrand
- `description`: không có mô tả, vấn đề này không quá ảnh hưởng đến quá trình phân tích

#### 4. Lọc dữ liệu

Check những dữ liệu có thể sai, cần chuẩn hóa sơ bộ

### `df_review`

#### 1. Thống kê cơ bản

In [17]:
df_review.shape

(158126, 8)

In [18]:
df_review.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158126 entries, 0 to 158125
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   review_id        158126 non-null  int64 
 1   product_id       158126 non-null  int64 
 2   user_name        158126 non-null  object
 3   rating           158126 non-null  int64 
 4   review_text      158126 non-null  object
 5   like_count       158126 non-null  int64 
 6   review_date      158126 non-null  object
 7   source_category  158126 non-null  object
dtypes: int64(4), object(4)
memory usage: 9.7+ MB


Nhận xét ngắn:
- kiểu dữ liệu ntn
- cần ktra gì

#### 2. Check duplicate

In [19]:
df_review[df_review.duplicated(subset=["review_id"], keep=False)]

,review_id,product_id,user_name,rating,review_text,like_count,review_date,source_category


In [20]:
#xóa các dòng trùng lặp
df_review = df_review.drop_duplicates(subset=["review_id"], keep="first").reset_index(drop=True)

#kiểm tra lại
if df_review.duplicated(subset=["review_id"], keep=False).any():
    print("There are duplicate rows in the 'review_id' column.")
else:
    print("No duplicate rows found in the 'review_id' column.")

No duplicate rows found in the 'review_id' column.


#### 3. Check missing

In [21]:
# kiểm tra missing
print("Missing values in 'df_review':")
print(df_review["review_id"].isnull().sum())

Missing values in 'df_review':
0


#### 4. Lọc dữ liệu

### `df_store`

#### 1. Thống kê cơ bản

In [22]:
df_category.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 497 entries, 0 to 496
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   category_id      497 non-null    int64 
 1   category_name    497 non-null    object
 2   parent_category  492 non-null    object
 3   source_category  497 non-null    object
dtypes: int64(1), object(3)
memory usage: 15.7+ KB


Nhận xét ngắn:
- kiểu dữ liệu ntn
- cần ktra gì

#### 2. Check duplicate

In [23]:
df_category[df_category.duplicated(subset=["category_name"], keep=False)]

,category_id,category_name,parent_category,source_category
121,5976,Máy sấy quần áo,Thiết bị gia đình,diengiadung
166,3863,Máy sấy quần áo,Dien tu - Dien lanh,dientu_dienlanh


In [24]:
#xóa các dòng trùng lặp
df_category = df_category.drop_duplicates(subset=["category_name"], keep="first").reset_index(drop=True)

#kiểm tra lại
if df_category.duplicated(subset=["category_name"], keep=False).any():
    print("There are duplicate rows in the 'category_name' column.")
else:
    print("No duplicate rows found in the 'category_name' column.")

No duplicate rows found in the 'category_name' column.


#### 3. Check missing

In [25]:
# kiểm tra missing
print("Missing values in 'df_review':")
print(df_review.isnull().sum())

Missing values in 'df_review':
review_id          0
product_id         0
user_name          0
rating             0
review_text        0
like_count         0
review_date        0
source_category    0
dtype: int64


#### 4. Lọc dữ liệu

## II. Chuẩn hóa dữ liệu

### II.1. Kiểm tra mapping giữa các bảng

- `df_product.product_id` so với `df_review.product_id`
- `df_product.store_id` so với `df_store.store_id`
- `df_product.category_id` so với `df_category.category_id`

Mục tiêu: tìm ra các bản ghi không mapping được để sau này lưu ý khi tạo database.

In [ ]:
def print_missing(left_df, right_df, key, left_name, right_name):
    left_ids = set(left_df[key].dropna().unique())
    right_ids = set(right_df[key].dropna().unique())
    
    missing = left_ids - right_ids
    
    if len(missing) > 0:
        print(f"\n=== {key}: có ở {left_name} nhưng KHÔNG có ở {right_name} ===")
        print(f"Số lượng: {len(missing)}")
        print("Ví dụ (tối đa 10):", list(missing)[:10])
        
        # show full row cho dễ debug
        display(left_df[left_df[key].isin(list(missing)[:10])])
    else:
        print(f"\n=== {key}: OK ({left_name} ⊆ {right_name}) ===")


# 1. reviews → products
print_missing(df_review, df_product, "product_id", "reviews", "products")

# 2. products → stores
print_missing(df_product, df_store, "store_id", "products", "stores")

# 3. products → categories
print_missing(df_product, df_category, "category_id", "products", "categories")


=== product_id: OK (reviews ⊆ products) ===

=== store_id: OK (products ⊆ stores) ===

=== category_id: OK (products ⊆ categories) ===


### II.2. Kiểm tra product thuộc nhiều hơn 1 category

Mục tiêu: tìm những product (theo `product_id`) đang gắn với từ 2 `category_id` trở lên → có khả năng lỗi gán category hoặc trùng dòng không nhất quán.

In [27]:
# === Product thuộc nhiều category ===

tmp = df_product[["product_id", "category_id"]].drop_duplicates()

multi_cat = (
    tmp.groupby("product_id")["category_id"]
    .nunique()
    .reset_index(name="category_count")
    .query("category_count > 1")
)

print("Số product thuộc nhiều category:", len(multi_cat))

if not multi_cat.empty:
    detail = multi_cat.merge(tmp, on="product_id")
    
    if "product_name" in df_product.columns:
        detail = detail.merge(
            df_product[["product_id", "product_name"]].drop_duplicates(),
            on="product_id",
            how="left"
        )
    
    display(detail.sort_values("category_count", ascending=False).head(50))

Số product thuộc nhiều category: 0


### II.3. Kiểm tra lại category của sản phẩm (dựa trên nội dung text)

Ý tưởng đơn giản:
- Dùng tên sản phẩm (hoặc mô tả) để tách từ khóa
- Gom nhóm theo `category_id` (và/hoặc tên category)
- Đếm tần suất từ xuất hiện trong từng category → xem top từ khóa đặc trưng
- Dựa vào trực quan, phát hiện sản phẩm có tên không phù hợp với category.

(Đây là phiên bản đơn giản hơn tf-idf, dễ chạy và hiểu.)

Kiểm tra một số category

In [28]:
df_category[["category_id", "category_name"]].head(20)

,category_id,category_name
0,1882,Dien gia dung
1,1884,Đồ dùng nhà bếp
2,1892,Nồi điện các loại
3,1893,Nồi cơm điện
4,1918,Nồi cơm điện tử
5,1919,Nồi cơm nắp gài
6,1920,Nồi cơm nắp rời
7,2271,Nồi cơm dung tích lớn
8,1894,Nồi áp suất điện
9,1895,Nồi lẩu điện


 Check lại category bằng keyword từ product_name 

In [ ]:
# 1. Chuẩn bị text (đơn giản)
df = df_product[["product_id", "category_id", "product_name"]].dropna()
df["product_name"] = df["product_name"].str.lower()

# 2. Vectorize (đếm từ)
vectorizer = CountVectorizer(max_features=1000, stop_words="english")
X = vectorizer.fit_transform(df["product_name"])

# 3. Tạo DataFrame từ khóa
keywords_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
keywords_df["category_id"] = df["category_id"].values

# 4. Lấy top keyword mỗi category
top_keywords = (
    keywords_df.groupby("category_id")
    .sum()
    .apply(lambda x: x.sort_values(ascending=False).head(10).index.tolist(), axis=1)
)

# 5. Hiển thị
print("Top keywords mỗi category:")
display(top_keywords.head(10))

Top keywords mỗi category:


category_id
1794    [bảng, máy, tính, hàng, hãng, chính, wifi, tab...
1795    [hàng, chính, hãng, điện, thoại, 128gb, xiaomi...
1796    [chính, 4g, hãng, hàng, thoại, điện, 10, 16, d...
1803    [hàng, nghe, máy, nhạc, chính, hãng, bluetooth...
1813    [loa, hàng, chính, hãng, tính, vi, soundmax, m...
1819    [hàng, hãng, chính, cho, điện, thoại, iphone, ...
1820    [nhớ, thẻ, hàng, hãng, chính, micro, 10, sd, 1...
1828    [usb, hàng, hãng, chính, 128gb, ultra, type, g...
1831    [chuột, lót, hàng, hãng, chính, miếng, bàn, pa...
1832    [laptop, nhiệt, tản, hàng, đế, hãng, chính, ch...
dtype: object

Chuẩn bị phụ kiện

In [30]:
accessory_phone = [
    # bảo vệ
    "ốp", "ốp lưng", "case", "bao da", "flip cover", "cường lực", "kính", "miếng dán", "dán màn hình",
    
    # sạc
    "cáp", "dây sạc", "củ sạc", "adapter", "sạc nhanh", "sạc không dây", "dock sạc",
    
    # âm thanh
    "tai nghe", "earphone", "headphone", "airpods", "buds",
    
    # khác
    "giá đỡ", "kẹp điện thoại", "gậy selfie", "ring", "móc", "dây đeo",
    "pin dự phòng", "power bank", "sạc dự phòng",
    "sim", "khay sim", "que chọc sim"
]

accessory_laptop = [
    "chuột", "mouse", "bàn phím", "keyboard", "pad chuột",
    "dock", "hub", "usb hub", "type c hub",
    "cáp", "hdmi", "displayport", "vga",
    "adapter", "củ sạc", "sạc laptop",
    "quạt tản nhiệt", "đế tản nhiệt",
    "ram", "ssd", "hdd", "ổ cứng",
    "webcam", "micro", "loa",
    "balo", "túi chống sốc", "túi laptop"
]

accessory_appliance = [
    "remote", "điều khiển", "điều khiển từ xa",
    "board", "mạch", "bo mạch",
    "cảm biến", "sensor",
    "dây điện", "jack", "đầu nối",
    "ống", "ống dẫn", "ống nước",
    "van", "lọc", "lưới lọc",
    "block", "motor", "quạt",
    "bánh răng", "trục",
    "phụ kiện", "linh kiện", "spare", "replacement"
]

accessory_home = [
    "nắp", "nắp nồi", "vung",
    "ruột", "lòng nồi",
    "khay", "giỏ", "lưới",
    "đầu", "đầu hút", "đầu xay", 
    "đế", "chân đế",
    "dây nguồn", "phích cắm"
]

accessory_generic = [
    "phụ kiện", "linh kiện", "đồ thay thế",
    "replacement", "spare", "part",
    "combo phụ kiện", "bộ phụ kiện",
    "kit", "set", "bộ",
]

accessory_keywords = list(set(
    accessory_phone +
    accessory_laptop +
    accessory_appliance +
    accessory_home +
    accessory_generic
))

device_keywords = [
    "điện thoại", "iphone", "samsung",
    "laptop", "macbook", "máy tính", "pc",
    "máy giặt", "máy lạnh", "điều hòa",
    "tủ lạnh", "tivi",
    "máy", "bếp", "lò"
]

def count_kw(text, keywords):
    return sum(kw in text for kw in keywords)

### II.2.b TF-IDF cho từ khóa theo category

Sử dụng `TfidfVectorizer` (sklearn) trên tên/mô tả sản phẩm để:
- Xây dựng ma trận TF-IDF cho toàn bộ sản phẩm
- Tính TF-IDF trung bình của từng từ trong từng `category_label`
- In ra top từ khóa có TF-IDF cao nhất (đặc trưng nhất) cho mỗi category.

In [ ]:
# 1. Chuẩn bị data + text
df = df_product[["product_id", "category_id", "product_name", "brand", "description"]].copy()

df["text"] = (
    df["product_name"].fillna("") + " " +
    df["brand"].fillna("") + " " +
    df["description"].fillna("")
)

# clean text
df["text"] = (
    df["text"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)


df = df.dropna(subset=["category_id", "text"])

# 2. Lọc category ít data
valid_cats = df["category_id"].value_counts()
valid_cats = valid_cats[valid_cats > 5].index
df = df[df["category_id"].isin(valid_cats)]

# 3. TF-IDF (có bigram)
vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["text"])

# 4. Tính centroid mỗi category
cat_ids = df["category_id"].unique()
centroids = {
    cat: np.asarray(X[df["category_id"].values == cat].mean(axis=0)).reshape(1, -1)
    for cat in cat_ids
}

centroid_matrix = np.vstack([centroids[c] for c in cat_ids])


# 5. Similarity matrix
sim_matrix = cosine_similarity(X, centroid_matrix)

# 6. Predict category
best_idx = sim_matrix.argmax(axis=1)
df["category_id_predict"] = [cat_ids[i] for i in best_idx]

# similarity với category hiện tại
cat_id_to_idx = {c: i for i, c in enumerate(cat_ids)}

# convert category_id thành index tương ứng
current_idx = df["category_id"].map(cat_id_to_idx).values

# lấy similarity đúng vị trí
df["similarity"] = sim_matrix[np.arange(len(df)), current_idx]

# 7. Margin (độ chắc chắn)
top2 = np.sort(sim_matrix, axis=1)[:, -2:]
df["margin"] = top2[:, 1] - top2[:, 0]

# 8. Flag (kết hợp nhiều điều kiện)
flagged = df[
    (df["similarity"] < 0.05) &
    (df["margin"] < 0.1) &
    (df["category_id"] != df["category_id_predict"])
]

# 9. Map category name
cat_map = df_category[["category_id", "category_name"]].drop_duplicates()

flagged = flagged.merge(
    cat_map.rename(columns={
        "category_id": "category_id_current",
        "category_name": "category_name_current"
    }),
    left_on="category_id",
    right_on="category_id_current",
    how="left"
)

flagged = flagged.merge(
    cat_map.rename(columns={
        "category_id": "category_id_predict",
        "category_name": "category_name_predict"
    }),
    on="category_id_predict",
    how="left"
)

# 10. Output
# === OUTPUT TOP-K SẢN PHẨM NGHI NGỜ ===

# 1. mismatch
df["is_mismatch"] = df["category_id"] != df["category_id_predict"]

# 2. similarity với category predict
df["sim_pred"] = sim_matrix.max(axis=1)

# 3. detect type


df["accessory_score"] = df["text"].apply(lambda x: count_kw(x, accessory_keywords))
df["device_score"] = df["text"].apply(lambda x: count_kw(x, device_keywords))
df["is_pure_accessory"] = (
    (df["accessory_score"] > 0) &
    (df["device_score"] == 0)
)

# 4. RULE FILTER (loại false positive)
df["is_false_positive"] = (
    df["is_pure_accessory"] & 
    df["is_mismatch"]
)

# 5. score (càng thấp càng đáng nghi)
df["score"] = (
    df["similarity"] 
    - df["sim_pred"] 
    + 0.5 * df["margin"]
    + 0.05 * df["accessory_score"]  
)



# 6. candidate (đã filter)
candidates = df[
    (df["is_mismatch"]) &
    (~df["is_false_positive"]) &  # loại phụ kiện bị kéo sai
    (df["similarity"] < 0.2) &   # giữ quality
    (df["margin"] < 0.15)
].copy()

# 7. lấy top
top_k = 50
candidates = candidates.sort_values("score").head(top_k)

# 8. map category name
cat_map = df_category[["category_id", "category_name"]].drop_duplicates()

result = candidates.merge(
    cat_map.rename(columns={
        "category_id": "category_id_current",
        "category_name": "category_name_current"
    }),
    left_on="category_id",
    right_on="category_id_current",
    how="left"
)

result = result.merge(
    cat_map.rename(columns={
        "category_id": "category_id_predict",
        "category_name": "category_name_predict"
    }),
    on="category_id_predict",
    how="left"
)

# 9. select columns
result = result[[
    "product_id",
    "product_name",
    "category_id_current",
    "category_name_current",
    "category_id_predict",
    "category_name_predict",
    "similarity",
    "margin",
    "score"
]]

# 10. display
#pd.set_option("display.max_rows", 100)

print(f"Số product nghi ngờ (top {top_k}): {len(result)}")
display(result)

Số product nghi ngờ (top 50): 50


,product_id,product_name,category_id_current,category_name_current,category_id_predict,category_name_predict,similarity,margin,score
0,276135770,Máy rang hạt cà phê công nghệ gia nhiệt không ...,8728,Đồ dùng nhà bếp khác,24050,Máy xay cà phê,0.159401,0.116175,-0.529377
1,275061940,Máy rang hạt cà phê công nghệ gia nhiệt không ...,8728,Đồ dùng nhà bếp khác,24050,Máy xay cà phê,0.174231,0.117744,-0.517457
2,278907834,Máy rang hạt cà phê và các loại hạt tự động th...,8728,Đồ dùng nhà bếp khác,24050,Máy xay cà phê,0.154691,0.130388,-0.512842
3,276611166,MÁY RỬA CHÉN KAFF KF-BDWSI12.6 - Hàng Chính Hãng,8728,Đồ dùng nhà bếp khác,67223,Máy rửa chén bán âm,0.193503,0.044726,-0.496075
4,278279719,MÁY RỬA CHÉN KAFF KF-SBL775B NEW PLUS - Hàng...,8728,Đồ dùng nhà bếp khác,67223,Máy rửa chén bán âm,0.195415,0.028548,-0.462433
5,48583111,"BẢO VỆ TỦ LẠNH, THIẾT BỊ LẠNH VỚI STANDA BẢO V...",28892,"Phụ kiện, linh kiện điện lạnh khác",67194,Tủ lạnh mini,0.154094,0.039973,-0.461688
6,275906423,MÁY RỬA CHÉN KF-W45A1A401J. Hàng Chính Hãng,23818,Máy sấy chén,67223,Máy rửa chén bán âm,0.192771,0.017329,-0.458526
7,275705476,Vòi rửa chén bát inox 206806a (hàng nhập khẩu),8728,Đồ dùng nhà bếp khác,67219,Máy rửa chén độc lập,0.176884,0.039735,-0.447436
8,273168047,GIÁ TREO LY EURONOX EN10. Hàng Chính Hãng,5263,Bàn Phím Văn Phòng Có Dây,8070,Giá treo loa,0.054187,0.063488,-0.447221
9,276611207,MÁY RỬA CHÉN KF-W60C3A401L - Hàng Chính Hãng,23818,Máy sấy chén,67223,Máy rửa chén bán âm,0.177633,0.000475,-0.446653


Chúng em đã thử nghiệm qua các mức giá trị thì 50-100 top score đầu sẽ cho ra tỉ lệ chính xác cao nhất.
Khó khăn:
- Do hạn chế của phương pháp lọc cơ bản.
- Sự đa dạng trong tên sản phẩm.
- Có nhiều category tương tự nhau (Mà sản phẩm có thể thuộc về 1 trong n category đó một cách hợp lý)

- Lý do chọn phương pháp này: không cần công cụ phức tạp, thời gian chạy lâu.
- Độ chính xác tốt, tuy không xuất sắc như các phương pháp ứng dụng LLM nhưng vẫn cho ra được kết quả tốt. Có tính ứng dụng cao trong việc phân loại sp vào đúng cateogry

Sau khi duyệt lại list, đây là list các sản phẩm bị gán nhầm category:

# cái bảng

In [32]:
#gán đúng category lại cho các product bị mismatch